In [ ]:
import os
import sys
sys.path.append("..")   
from src.monitorizacion.drift import generar_report_drift
import pandas as pd


In [ ]:
snap = generar_report_drift()
d = snap.dict()
print(type(d))
print(d.keys())

In [ ]:
metrics = d["metrics"]
print("nº de entradas:", len(metrics))
print("--- primera entrada ---")
print(metrics[0])
print("--- segunda entrada ---")
print(metrics[1])

In [ ]:


filas = []
for m in metrics[1:]:                       
    cfg = m.get("config", {})
    col = cfg.get("column")
    if col is None:                        
        continue
    score = m["value"]
    if isinstance(score, dict):            
        continue
    umbral = cfg.get("threshold")
    filas.append({
        "columna": col,
        "score": float(score),
        "umbral": umbral,
        "drift": float(score) > umbral,
    })

tabla_drift = pd.DataFrame(filas).sort_values("score", ascending=False).reset_index(drop=True)
print("Columnas con drift:", tabla_drift["drift"].sum(), "de", len(tabla_drift))
print(tabla_drift.to_string())

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")
import pandas as pd
from src.procesamiento.concatenacion import construir_datetime_omie

# 1) Predicciones guardadas y precio real
reg  = pd.read_parquet("../data/monitoring/predicciones.parquet")
real = construir_datetime_omie(pd.read_csv("../data/interim/tabla_OMIE"))[["datetime_utc", "precio_espana"]]
ev = reg.merge(real, on="datetime_utc", how="inner")
ev["hora"] = ev["datetime_utc"].dt.tz_convert("Europe/Madrid").dt.hour
print("Horas con precio real:", len(ev))

# 2) Error de nivel (MAE) y de forma
mae_mod   = (ev["prediccion"] - ev["precio_espana"]).abs().mean()
mae_naive = (ev["naive_d1"]   - ev["precio_espana"]).abs().mean()
forma = lambda p: ((ev[p] - ev[p].mean()) - (ev["precio_espana"] - ev["precio_espana"].mean())).abs().mean()
print(f"MAE      → modelo {mae_mod:.2f} | naive {mae_naive:.2f}")
print(f"Forma    → modelo {forma('prediccion'):.2f} | naive {forma('naive_d1'):.2f}")
print(f"Spearman → modelo {ev['prediccion'].corr(ev['precio_espana'], method='spearman'):.3f} | "
      f"naive {ev['naive_d1'].corr(ev['precio_espana'], method='spearman'):.3f}")

# 3) ¿Acierta las horas baratas y caras? (K = 1, 2 y 4)
for K in [1, 2, 4]:
    for col in ["prediccion", "naive_d1"]:
        pb = ev.nsmallest(K, col)   # horas que el modelo cree más baratas
        pc = ev.nlargest(K, col)    # horas que el modelo cree más caras
        rb = ev.nsmallest(K, "precio_espana")
        rc = ev.nlargest(K, "precio_espana")
        coste_compra = pb["precio_espana"].mean() - rb["precio_espana"].mean()   # cuánto pagas de más
        coste_venta  = rc["precio_espana"].mean() - pc["precio_espana"].mean()   # cuánto dejas de cobrar
        print(f"K={K} {col:<11} compra: {sorted(pb['hora'])} vs reales {sorted(rb['hora'])} → +{coste_compra:.1f} €/MWh | "
              f"venta: {sorted(pc['hora'])} vs reales {sorted(rc['hora'])} → -{coste_venta:.1f} €/MWh")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # raíz del proyecto (el notebook está en notebooks/)

%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from src.monitorizacion.evaluacion import cruzar_con_real, ROOT

tabla = pd.read_parquet(ROOT / "data/processed/tabla_features.parquet")
print(tabla.loc[tabla["precio_espana"].notna(), "datetime_utc"].max())

registro = pd.read_parquet(ROOT / "data/monitoring/predicciones_prueba.parquet")
cruzado = cruzar_con_real(registro, tabla)
cruzado["fecha_objetivo"].value_counts()

In [ ]:
print(registro["fecha_objetivo"].unique())
print(registro["datetime_utc"].dtype, "|", tabla["datetime_utc"].dtype)
print(tabla.loc[tabla["precio_espana"].notna(), "datetime_utc"].max())

In [ ]:
evaluar(ROOT / "data/monitoring/predicciones_prueba.parquet")

In [ ]:
from src.monitorizacion.evaluacion import cruzar_con_real, metricas_dia
cruzado = cruzar_con_real(registro, tabla)
metricas_dia(cruzado)


In [ ]:
print(cruzado["prediccion"].std(), cruzado["precio_real"].std())

In [ ]:
from src.monitorizacion.evaluacion import regret_dia
for k in (1, 2, 4):
    print(k, regret_dia(cruzado, k).to_dict())

In [ ]:
from src.monitorizacion.evaluacion import evaluar
evaluar(ROOT / "data/monitoring/predicciones_prueba.parquet")
